In [10]:
import opendatasets as od

od.download("https://www.kaggle.com/competitions/home-credit-default-risk/data", download_dir="./data/inputs")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username:Your Kaggle Key:Downloading home-credit-default-risk.zip to .\home-credit-default-risk


100%|██████████| 688M/688M [00:24<00:00, 29.7MB/s] 



Extracting archive .\home-credit-default-risk/home-credit-default-risk.zip to .\home-credit-default-risk


In [42]:
import pandas as pd 


### Merging Data

In [66]:
bureau_df = pd.read_csv("../data/inputs/home-credit-default-risk/bureau.csv")
raw_train_df = pd.read_csv("../data/inputs/home-credit-default-risk/application_train.csv").dropna(subset=["TARGET"])
raw_test_df = pd.read_csv("../data/inputs/home-credit-default-risk/application_test.csv")

len(bureau_df), len(raw_train_df), len(raw_test_df)

(1716428, 307511, 48744)

In [67]:
raw_train_df['TARGET'].value_counts()

TARGET
0    282686
1     24825
Name: count, dtype: int64

In [ ]:
bureau_agg = bureau_df.groupby("SK_ID_CURR").agg({
    "SK_ID_BUREAU": "count",                          # number of credits
    "AMT_CREDIT_SUM": ["mean", "sum"],                # exposure
    "AMT_CREDIT_SUM_DEBT": "sum",                     # total debt
    "AMT_CREDIT_SUM_OVERDUE": "sum",                  # total overdue
    "CREDIT_DAY_OVERDUE": "max",                      # worst delay
    "DAYS_CREDIT": "mean"                             # recency
}).rename(columns={
    "SK_ID_BUREAU": "bureau_count",
    "AMT_CREDIT_SUM": "avg_credit_sum",
    "AMT_CREDIT_SUM_DEBT": "total_debt",
    "AMT_CREDIT_SUM_OVERDUE": "total_overdue",
    "CREDIT_DAY_OVERDUE": "max_overdue",
    "DAYS_CREDIT": "avg_credit_days"
})

In [72]:
train = raw_train_df.merge(bureau_agg, on="SK_ID_CURR", how="left")
test  = raw_test_df.merge(bureau_agg, on="SK_ID_CURR", how="left")

In [71]:
bureau_df.columns

Index(['SK_ID_CURR', 'SK_ID_BUREAU', 'CREDIT_ACTIVE', 'CREDIT_CURRENCY',
       'DAYS_CREDIT', 'CREDIT_DAY_OVERDUE', 'DAYS_CREDIT_ENDDATE',
       'DAYS_ENDDATE_FACT', 'AMT_CREDIT_MAX_OVERDUE', 'CNT_CREDIT_PROLONG',
       'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT', 'AMT_CREDIT_SUM_LIMIT',
       'AMT_CREDIT_SUM_OVERDUE', 'CREDIT_TYPE', 'DAYS_CREDIT_UPDATE',
       'AMT_ANNUITY'],
      dtype='str')